<a href="https://colab.research.google.com/github/NicoleLeiva/ExamenLenguajes2/blob/main/03_corte3_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

REPO = "NicoleLeiva/examen1_info_Nicole"
TOKEN = userdata.get("GITHUB_TOKEN")

!rm -rf /content/repo
!git clone -q https://{TOKEN}@github.com/{REPO}.git /content/repo
%cd /content/repo/scripts

/content/repo/scripts


Celda 1, descargar el repositorio

Celda 2, construir la tabla de modelado

In [2]:
import pandas as pd

d = pd.read_csv("../datos/processed/caso_02_pedidos_limpio.csv", parse_dates=["timestamp"])
d["fecha_prometida"] = pd.to_datetime(d["fecha_prometida"])

def primero(a):
    return d[d.activity == a].groupby("case_id")["timestamp"].min()

g = d.groupby("case_id")
c = g[["categoria", "metodo_pago", "region", "transportista", "canal_venta",
       "monto", "cantidad_productos", "fecha_prometida"]].first()
c["recibido"] = primero("Pedido recibido")
c["despacho"] = primero("Despacho")
c["entrega"] = d[d.activity == "Entrega"].groupby("case_id")["timestamp"].max()

# Solo eventos ocurridos hasta el Despacho (momento de predicción)
d = d.join(c["despacho"].rename("t_despacho"), on="case_id")
antes = d[d["timestamp"] <= d["t_despacho"]]

for nombre, act in [("falta_stock", "Falta de stock"),
                    ("recordatorio", "Recordatorio de pago"),
                    ("correccion_direccion", "Corrección de dirección")]:
    c[nombre] = (antes[antes.activity == act].groupby("case_id").size()
                 .reindex(c.index).fillna(0).gt(0).astype(int))

c["n_actividades_hasta_despacho"] = antes.groupby("case_id").size()
c["horas_hasta_despacho"] = (c["despacho"] - c["recibido"]).dt.total_seconds() / 3600
c["dias_prometidos"] = (c["fecha_prometida"] - c["recibido"].dt.normalize()).dt.days
c["dias_margen_al_despacho"] = (c["fecha_prometida"] - c["despacho"].dt.normalize()).dt.days

# Población: pedidos entregados. Variable objetivo: llegó después de la fecha prometida
m = c.dropna(subset=["entrega", "despacho"]).copy()
m["atrasado"] = (m["entrega"].dt.normalize() > m["fecha_prometida"]).astype(int)

print("Pedidos en la tabla:", len(m))
print("Atrasados:", int(m["atrasado"].sum()), "|", round(m["atrasado"].mean() * 100, 2), "%")
print("Valores vacíos:", int(m.isna().sum().sum()))
m.head()


Pedidos en la tabla: 1869
Atrasados: 559 | 29.91 %
Valores vacíos: 0


,categoria,metodo_pago,region,transportista,canal_venta,monto,cantidad_productos,fecha_prometida,recibido,despacho,entrega,falta_stock,recordatorio,correccion_direccion,n_actividades_hasta_despacho,horas_hasta_despacho,dias_prometidos,dias_margen_al_despacho,atrasado
case_id,,,,,,,,,,,,,,,,,,,
PED00002,Hogar,Transferencia,Central,LogiSur,Teléfono,204.72,1,2026-04-04,2026-03-31 15:38:51,2026-04-06 15:00:01,2026-04-08 08:00:00,0,0,0,6.0,143.352778,4,-2.0,1
PED00003,Electrónica,Tarjeta,Caribe,TransRápido,App,699.00,1,2026-05-07,2026-05-01 14:17:34,2026-05-04 15:00:01,2026-05-06 08:00:00,1,0,0,8.0,72.707500,6,3.0,0
PED00004,Muebles,Tarjeta,Central,TransRápido,App,376.63,2,2026-06-14,2026-06-10 17:48:58,2026-06-15 15:00:01,2026-06-17 08:00:00,1,0,0,8.0,117.184167,4,-1.0,1
PED00005,Hogar,Tarjeta,Central,EnvíaYa,App,122.92,3,2026-04-12,2026-04-08 07:16:08,2026-04-08 15:00:01,2026-04-09 11:29:13,0,0,0,6.0,7.731389,4,4.0,0
PED00006,Hogar,Tarjeta,Sur,LogiSur,Web,58.54,2,2026-04-05,2026-03-30 17:11:22,2026-03-31 15:00:01,2026-04-04 08:08:00,0,0,0,6.0,21.810833,6,5.0,0


Celda 3: separar los datos y definir el modelo base

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)

cat = ["categoria", "metodo_pago", "region", "transportista", "canal_venta"]
num = ["monto", "cantidad_productos", "falta_stock", "recordatorio", "correccion_direccion",
       "n_actividades_hasta_despacho", "horas_hasta_despacho", "dias_prometidos",
       "dias_margen_al_despacho"]

X, y = m[cat + num], m["atrasado"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Entrenamiento:", len(Xtr), "| Prueba:", len(Xte))
print("% atrasado en entrenamiento:", round(ytr.mean() * 100, 2), "| en prueba:", round(yte.mean() * 100, 2))

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                         ("num", StandardScaler(), num)])

def evalua(nombre, modelo):
    p = modelo.predict(Xte)
    try:
        auc = roc_auc_score(yte, modelo.predict_proba(Xte)[:, 1])
    except Exception:
        auc = float("nan")
    print(f"{nombre}: exactitud={accuracy_score(yte, p):.3f} precision={precision_score(yte, p, zero_division=0):.3f} "
          f"recall={recall_score(yte, p):.3f} F1={f1_score(yte, p):.3f} AUC={auc:.3f}")
    print("Matriz de confusión (filas = real, columnas = predicho; 0 = a tiempo, 1 = atrasado):")
    print(confusion_matrix(yte, p))

base = Pipeline([("pre", pre), ("m", DummyClassifier(strategy="most_frequent"))]).fit(Xtr, ytr)
evalua("MODELO BASE (siempre 'a tiempo')", base)

Entrenamiento: 1495 | Prueba: 374
% atrasado en entrenamiento: 29.9 | en prueba: 29.95
MODELO BASE (siempre 'a tiempo'): exactitud=0.701 precision=0.000 recall=0.000 F1=0.000 AUC=0.500
Matriz de confusión (filas = real, columnas = predicho; 0 = a tiempo, 1 = atrasado):
[[262   0]
 [112   0]]


Celda 4: entrenar los dos modelos

In [5]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)

lr = GridSearchCV(
    Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    {"m__C": [0.01, 0.1, 1, 10]}, cv=cv, scoring="f1").fit(Xtr, ytr)
print("Regresión logística, mejores parámetros:", lr.best_params_, "| F1 en validación cruzada:", round(lr.best_score_, 3))
evalua("REGRESIÓN LOGÍSTICA", lr)

rf = GridSearchCV(
    Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                         random_state=42, n_jobs=-1))]),
    {"m__max_depth": [4, 6, 10, None], "m__min_samples_leaf": [1, 5, 10]},
    cv=cv, scoring="f1").fit(Xtr, ytr)
print("\nBosque aleatorio, mejores parámetros:", rf.best_params_, "| F1 en validación cruzada:", round(rf.best_score_, 3))
evalua("BOSQUE ALEATORIO", rf)

Regresión logística, mejores parámetros: {'m__C': 10} | F1 en validación cruzada: 0.723
REGRESIÓN LOGÍSTICA: exactitud=0.805 precision=0.636 recall=0.812 F1=0.714 AUC=0.889
Matriz de confusión (filas = real, columnas = predicho; 0 = a tiempo, 1 = atrasado):
[[210  52]
 [ 21  91]]

Bosque aleatorio, mejores parámetros: {'m__max_depth': 6, 'm__min_samples_leaf': 1} | F1 en validación cruzada: 0.731
BOSQUE ALEATORIO: exactitud=0.856 precision=0.754 recall=0.768 F1=0.761 AUC=0.917
Matriz de confusión (filas = real, columnas = predicho; 0 = a tiempo, 1 = atrasado):
[[234  28]
 [ 26  86]]


Celda 5: qué variables importan

In [6]:
from sklearn.inspection import permutation_importance

for nombre, modelo in [("Regresión logística", lr), ("Bosque aleatorio", rf)]:
    imp = permutation_importance(modelo.best_estimator_, Xte, yte, scoring="f1", n_repeats=10, random_state=42)
    s = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False)
    print(f"\n--- Importancia por permutación: {nombre} ---")
    print(s.round(3).head(8).to_string())


--- Importancia por permutación: Regresión logística ---
dias_margen_al_despacho    0.249
dias_prometidos            0.092
transportista              0.028
region                     0.012
horas_hasta_despacho       0.011
recordatorio               0.006
categoria                  0.005
falta_stock                0.001

--- Importancia por permutación: Bosque aleatorio ---
horas_hasta_despacho            0.190
transportista                   0.052
dias_margen_al_despacho         0.032
monto                           0.014
metodo_pago                     0.014
n_actividades_hasta_despacho    0.011
categoria                       0.009
region                          0.008


verificar cifras

In [7]:
m["entrega_fallida"] = (d[d.activity == "Entrega fallida"].groupby("case_id").size()
                        .reindex(m.index).fillna(0).gt(0).astype(int))
p = rf.predict(Xte)
t = Xte.copy(); t["y"] = yte; t["p"] = p; t["fallida"] = m.loc[Xte.index, "entrega_fallida"]
fn = t[(t.y == 1) & (t.p == 0)]; tp = t[(t.y == 1) & (t.p == 1)]

print("Falsos negativos:", len(fn), "| con entrega fallida:", int(fn.fallida.sum()), "| %:", round(fn.fallida.mean()*100, 1))
print("Detectados:", len(tp), "| con entrega fallida:", int(tp.fallida.sum()), "| %:", round(tp.fallida.mean()*100, 1))
print("Horas hasta despacho -> FN:", round(fn.horas_hasta_despacho.mean(), 1), "| detectados:", round(tp.horas_hasta_despacho.mean(), 1))
print("Margen de días -> FN:", round(fn.dias_margen_al_despacho.mean(), 2), "| detectados:", round(tp.dias_margen_al_despacho.mean(), 2))
print(m.groupby("metodo_pago")["horas_hasta_despacho"].mean().round(1).to_string())
print(m.groupby("falta_stock")["horas_hasta_despacho"].mean().round(1).to_string())
print(m.groupby("recordatorio")["horas_hasta_despacho"].mean().round(1).to_string())
print("Margen <= 0:", int((m.dias_margen_al_despacho <= 0).sum()), "| atrasados:", int(m[m.dias_margen_al_despacho <= 0].atrasado.sum()))
print("Margen >= 1:", int((m.dias_margen_al_despacho >= 1).sum()), "| atrasados:", int(m[m.dias_margen_al_despacho >= 1].atrasado.sum()))

Falsos negativos: 26 | con entrega fallida: 8 | %: 30.8
Detectados: 86 | con entrega fallida: 9 | %: 10.5
Horas hasta despacho -> FN: 45.8 | detectados: 107.7
Margen de días -> FN: 3.46 | detectados: 0.58
metodo_pago
Contra entrega    45.9
Tarjeta           46.6
Transferencia     86.0
falta_stock
0     49.3
1    111.3
recordatorio
0     53.5
1    107.0
Margen <= 0: 205 | atrasados: 205
Margen >= 1: 1664 | atrasados: 354
